# Operations Reasearch: Portfolio Optimization Example (nonlinear program)

**Pre-requisites:**
- Gurobi and Python installed
- Make sure that an academic license for Gurobi is applied and activated.


**Description:**

We have an opportunity to invest in three different stocks, but our
budget is limited.
- Each stock has its own current price, expected future price, and variance of future price.
- Our goal is to decide whether and how much we should invest in each stock in order to minimize the variance (i.e. risk) of total revenue while ensuring the expected revenue is high enough.

## Formulation
$$
\begin{aligned}
\min \quad & \sum_{i=1}^{n} \sigma_i^2 x_i^2 \\
\text{s.t.} \quad & \sum_{i=1}^{n} p_i x_i \leq B \\
& \sum_{i=1}^{n} u_i x_i \geq R \\
& x_i \geq 0 \qquad \forall\, i = 1, \ldots, n
\end{aligned}
$$

, where $\sigma_i^2$ is the variance of the stock $x_i$
$p_i$ is the price, and $u_i$ is the expected price

*Constraints:*

- The First Constraint is the Budget Constraint
- The Second Constraint is the Revenue Guarantee

In [33]:
from gurobipy import *
import pandas as pd
from pathlib import Path
#get current working directory
chwd = Path.cwd()
data_dir = chwd / ".." / "data"

In [34]:
stock_info = pd.read_excel(data_dir / 'NLP_dataset.xlsx', 'Stock information')
stocks = range(len(stock_info['Stock']))
prices = stock_info['Price']
exp_prices = stock_info['Expected price']
variances = stock_info['Variance of the price']

other_info = pd.read_excel(data_dir / 'NLP_dataset.xlsx', 'Budget and min_exp_profit')
budget = other_info['Budget'].iloc[0]
min_exp_rev = other_info['Minimum expected profit'].iloc[0]

In [35]:
eg3 = Model("eg3")    # build a new model
    
# add variables as a list
x = []
for i in stocks:
    x.append(eg3.addVar(lb = 0, vtype = GRB.CONTINUOUS, name = "x" + str(i+1)))
                 
# setting the objective function 
eg3.setObjective(quicksum((variances[i] * x[i] * x[i]) for i in stocks), GRB.MINIMIZE) 

# add constraints and name them
eg3.addConstr(quicksum(prices[i] * x[i] for i in stocks) <= budget, "budget_limit")
eg3.addConstr(quicksum(exp_prices[i] * x[i] for i in stocks) >= min_exp_rev, "min_revenue")
    
eg3.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M5
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 2 rows, 3 columns and 6 nonzeros (Min)
Model fingerprint: 0x84ab935d
Model has 0 linear objective coefficients
Model has 3 quadratic objective terms
Coefficient statistics:
  Matrix range     [2e+01, 6e+01]
  Objective range  [0e+00, 0e+00]
  QObjective range [2e+02, 3e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+05, 1e+05]

Presolve time: 0.01s
Presolved: 2 rows, 3 columns, 6 nonzeros
Presolved model has 3 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 1.000e+00
 Factor NZ  : 3.000e+00
 Factor Ops : 5.000e+00 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl     Time
   0   2.03097694e+09 -2.03097694e+09  2.34e+03

In [36]:
for i in stocks:
    print(x[i].varName, '=', x[i].x)

print("z* =", eg3.objVal)    # print objective value

print("Expected profit =", sum(exp_prices[i] * x[i].x for i in stocks))
print("Total spending =", sum(prices[i] * x[i].x for i in stocks))

x1 = 1333.333333333334
x2 = 833.3333333333323
x3 = 2.6356475320356435e-15
z* = 1288888888.8888865
Expected profit = 115000.0
Total spending = 100000.0


***Exercise*** Suppose that our goal becomes maximizing total profit, given an upper bound of the risk. How should we modify the code?